# Classifier evaluation
Cross-validated accuracy, per-class report and confusion matrix for the TF-IDF classifier on the seed data.
Run from the repo root's `notebooks/` folder with the project requirements installed.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from app.classifier import build_pipeline

data = pd.read_csv('../data/training.csv')
data['category'].value_counts()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
pred = cross_val_predict(build_pipeline(), data['text'], data['category'], cv=cv)
print(classification_report(data['category'], pred))

In [ ]:
labels = sorted(data['category'].unique())
pd.DataFrame(confusion_matrix(data['category'], pred, labels=labels), index=labels, columns=labels)

## Most informative terms per category

In [ ]:
import numpy as np
model = build_pipeline().fit(data['text'], data['category'])
vec, lr = model.steps[0][1], model.steps[1][1]
terms = np.array(vec.get_feature_names_out())
for i, c in enumerate(lr.classes_):
    print(f'{c:15s}', ', '.join(terms[np.argsort(lr.coef_[i])[-8:][::-1]]))

## Confidence calibration
If low-confidence predictions are also the wrong ones, the confidence indicator in the UI is trustworthy.

In [ ]:
proba = cross_val_predict(build_pipeline(), data['text'], data['category'], cv=cv, method='predict_proba')
conf = proba.max(axis=1)
correct = pred == data['category'].values
pd.DataFrame({'confidence': conf, 'correct': correct}).groupby(pd.cut(conf, [0, .4, .7, 1.0]), observed=True)['correct'].agg(['mean', 'size'])